# Deployment Patterns & Rollbacks

How you release a model is as important as what you release. This note covers shadow, canary, blue-green, and A/B rollout patterns with a simulated canary analysis, plus rollback triggers and model+data schema compatibility.

## What Interviewers Test
- Shadow vs canary vs blue-green vs A/B — risk, cost, what each catches
- Implementing a sequential canary analysis
- Feature flag patterns for model versioning
- Rollback triggers: what metrics, what thresholds
- Schema compatibility: what breaks when features change

## Deployment Patterns Comparison

| Pattern | Traffic split | Risk | What it catches | Cost |
|---|---|---|---|---|
| **Shadow** | 100% control, 100% shadow (no serving) | Very low | Latency, errors, output distribution | 2× infra cost |
| **Canary** | 1–10% new, 90–99% old | Low | Real user impact at small scale | Low overhead |
| **Blue-green** | 100% cutover to new | Medium | Full scale behavior; instant rollback | 2× infra briefly |
| **A/B test** | 50/50 or other split | Medium | Statistical significance on metrics | A/B infrastructure |

> 💡 **Interview Tip:** Shadow testing is the safest option and what most MLEs recommend for high-stakes models. It lets you compare outputs in production without any user impact. Follow up: *"What can shadow testing NOT catch?"* Answer: personalization effects, feedback loops, novelty effects — anything that requires real user responses.


In [ ]:
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Canary analysis: sequential test ---
def canary_analysis(control_metric, treatment_metric, alpha=0.05):
    """
    Compare treatment vs control metrics.
    Returns whether to promote, reject, or continue.
    """
    n_c, n_t = len(control_metric), len(treatment_metric)
    mean_c, mean_t = control_metric.mean(), treatment_metric.mean()
    relative_change = (mean_t - mean_c) / (mean_c + 1e-12)
    
    # Two-sample t-test
    t_stat, p_value = stats.ttest_ind(treatment_metric, control_metric)
    
    # Effect size (Cohen's d)
    pooled_std = np.sqrt((control_metric.var() + treatment_metric.var()) / 2)
    cohens_d = (mean_t - mean_c) / (pooled_std + 1e-12)
    
    return {
        'n_control':       n_c,
        'n_treatment':     n_t,
        'control_mean':    mean_c,
        'treatment_mean':  mean_t,
        'relative_change': relative_change,
        'p_value':         p_value,
        'cohens_d':        cohens_d,
        'significant':     p_value < alpha,
        'direction':       'better' if mean_t > mean_c else 'worse',
    }

# Simulate canary: new model is slightly better
n_users_per_arm = 1000
control_ctr    = np.random.binomial(1, 0.08, n_users_per_arm).astype(float)  # 8% CTR
treatment_ctr  = np.random.binomial(1, 0.085, n_users_per_arm).astype(float) # 8.5% CTR

result = canary_analysis(control_ctr, treatment_ctr)
print("=== Canary Analysis ===")
print(f"Control CTR:   {result['control_mean']:.4f}")
print(f"Treatment CTR: {result['treatment_mean']:.4f}")
print(f"Relative change: {result['relative_change']*100:+.2f}%")
print(f"p-value: {result['p_value']:.4f}, significant: {result['significant']}")
print(f"Cohen's d: {result['cohens_d']:.4f}")
print()
decision = "PROMOTE" if result['significant'] and result['direction'] == 'better' else "CONTINUE" if not result['significant'] else "ROLLBACK"
print(f"Decision: {decision}")


In [ ]:
# --- Schema compatibility demo: what breaks when features change ---
import json

# Version 1: original feature schema
SCHEMA_V1 = {
    'user_age': 'float',
    'item_category': 'int',
    'session_length': 'float',
    'time_of_day': 'float',
}

# Version 2: added features, renamed one, dropped another
SCHEMA_V2 = {
    'user_age':        'float',       # same
    'item_category':   'int',         # same
    'session_duration': 'float',      # RENAMED from session_length → BREAKING
    'time_of_day':     'float',       # same
    'device_type':     'int',         # NEW — needs default or retrain
    # 'time_of_day' might be removed in v3
}

def check_schema_compatibility(schema_train, schema_serve):
    issues = []
    # Features in training but not serving (missing at serve time)
    for f in schema_train:
        if f not in schema_serve:
            issues.append(f"MISSING at serve: '{f}' — will get NaN or error")
    # Features in serving but not training (unexpected)
    for f in schema_serve:
        if f not in schema_train:
            issues.append(f"NEW at serve: '{f}' — model never saw this feature")
    # Type mismatches
    for f in schema_train:
        if f in schema_serve and schema_train[f] != schema_serve[f]:
            issues.append(f"TYPE MISMATCH: '{f}' was {schema_train[f]}, now {schema_serve[f]}")
    return issues

issues = check_schema_compatibility(SCHEMA_V1, SCHEMA_V2)
print("Schema compatibility check (V1 model → V2 serving):")
for issue in issues:
    print(f"  ⚠️  {issue}")
if not issues:
    print("  ✓ Fully compatible")


## Rollback Triggers

| Trigger | Threshold | Action |
|---|---|---|
| Error rate | > 2× baseline | Immediate automated rollback |
| Latency p99 | > SLA (e.g., 200ms) | Immediate automated rollback |
| Business metric | > 5% drop vs control | Manual review then rollback |
| Model score distribution | PSI > 0.3 | Alert + manual review |
| Null rate on features | > 2× baseline | Alert + investigate |

**Rollback playbook:**
1. Detect anomaly via alert
2. Verify in monitoring dashboard (not a fluke)
3. One-command rollback to previous version (blue-green makes this instant)
4. Post-mortem: root cause analysis within 24 hours
5. Fix and re-deploy through canary


## Common Interview Questions

**Q: What is the difference between blue-green and canary deployment?**
Blue-green switches 100% of traffic from old (blue) to new (green) all at once, but keeps the old version running for instant rollback. Canary gradually shifts traffic (1%, 5%, 10%, ...) to the new version while monitoring metrics at each step. Canary exposes issues at small scale before full rollout; blue-green is faster to fully deploy but riskier.

**Q: What can shadow testing NOT catch that canary can?**
Shadow testing runs the new model in parallel but doesn't serve its outputs to users. It can catch latency, errors, and output distribution changes, but not effects that require real user interaction: novelty effects, personalization feedback, changes in user behavior caused by the new recommendations.

**Q: What is a feature flag and how does it apply to model deployment?**
A feature flag is a runtime configuration that controls which code path or model version a request follows, without redeployment. For ML models, flags enable gradual rollout (route X% of traffic to new model), instant rollback (flip flag to 0%), and A/B tests without code deploys. The flag value can be per-user, per-experiment, or global.

**Q: What happens when you add a new feature to a deployed model?**
If you add a feature to the training schema without updating the serving code, the model receives unexpected inputs (or raises an error). If you update serving before retraining, the model has never seen the new feature and will produce garbage outputs. Always coordinate schema changes: update serving to handle both old and new schema, retrain model, then cut over.

## Key Takeaways
- Shadow: safest (no user impact); canary: catches user effects at scale; blue-green: instant rollback
- Canary analysis: compare means with t-test, check p-value and effect size, decision = promote/continue/rollback
- Schema compatibility: check for missing features (serve-time error), renamed features (missing), type mismatches
- Rollback triggers: error rate, latency, business metric, score distribution — automate the first two
- Feature flags enable zero-downtime rollback and gradual traffic shifts
- Post-mortem every rollback: root cause → fix → re-deploy through canary